# Music Annotation Project V8.8D
## Notebook 55: Notebook 09 Shadow Comparison

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook compares the successfully reproduced Notebook 51R2 historical registered-replay path with the incumbent Notebook 09 path on the same page roster. The replay includes historical decision-code-2 exclusion solely because that is required to reproduce the registered 51R2 golden output. This does not authorise SAFE_PROTECT operationally.

### Entry conditions
- Notebook 53 passed.
- Notebook 54 passed the strict zero-tolerance equivalence audit.
- A completed Notebook 09 run exists for the same pages, or an explicit incumbent page directory is supplied.

### Fixed rules
- No retraining, recalibration, threshold movement or operational SAFE_PROTECT authorisation.
- Page-roster identity is mandatory.
- Outside-authority changes are forbidden.
- Disagreement is evidence for review, not permission to choose the more attractive output.
- Every failure is written before the notebook stops.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib pyarrow


In [ ]:
import cv2
import json
import hashlib
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime, timezone


ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

N53_ROOT = (
    ROOT
    / 'observed_print_classifier_v8_8_notebook53R3'
)

N54_ROOT = (
    ROOT
    / 'm5_remove_exact_equivalence_v8_8c'
)

N09_ROOT = (
    ROOT
    / 'production_pilot'
    / 'runs'
)

OUTPUT_ROOT = (
    ROOT
    / 'registered_replay_notebook09_shadow_v8_8d'
)

NOTEBOOK53_RUN_OVERRIDE = None
NOTEBOOK54_RUN_OVERRIDE = None
NOTEBOOK09_RUN_OVERRIDE = None
INCUMBENT_PAGE_DIR_OVERRIDE = None


def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def read_json(path):
    path = Path(path)

    require(
        path.exists(),
        f'Missing JSON file: {path}'
    )

    try:
        value = json.loads(path.read_text())
    except Exception as error:
        raise RuntimeError(
            f'Could not read JSON file {path}: {error}'
        ) from error

    require(
        isinstance(value, dict),
        f'JSON root must be an object: {path}'
    )

    return value


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = Path(str(path) + '.tmp')

    temporary.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            default=str,
        )
    )

    temporary.replace(path)


def sha256_file(path):
    path = Path(path)

    require(
        path.exists() and path.is_file(),
        f'Missing file for SHA-256: {path}'
    )

    digest = hashlib.sha256()

    with open(path, 'rb') as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b'',
        ):
            digest.update(block)

    return digest.hexdigest()


def array_hash(array):
    array = np.ascontiguousarray(array)

    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode())
    digest.update(str(array.shape).encode())
    digest.update(array.tobytes())

    return digest.hexdigest()


def latest_complete(root, required):
    root = Path(root)
    hits = []

    if root.exists():
        for path in root.rglob('*'):
            if not path.is_dir():
                continue

            if all((path / item).exists() for item in required):
                hits.append(path)

    if not hits:
        return None

    return max(
        hits,
        key=lambda path: path.stat().st_mtime
    )


def imread(path, flags=cv2.IMREAD_GRAYSCALE):
    path = Path(path)

    require(
        path.exists() and path.is_file(),
        f'Missing image: {path}'
    )

    image = cv2.imread(str(path), flags)

    require(
        image is not None,
        f'Unreadable image: {path}'
    )

    return image


def page_files(folder):
    folder = Path(folder)

    supported = {
        '.png',
        '.jpg',
        '.jpeg',
        '.tif',
        '.tiff',
        '.bmp',
    }

    if not folder.exists():
        return {}

    return {
        path.stem: path
        for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in supported
    }


RUN = (
    datetime.now().strftime('%Y%m%d_%H%M%S')
    + '_notebook55_v8_8d_shadow'
)

OUT = OUTPUT_ROOT / RUN
T = OUT / 'tables'
D = OUT / 'differences'
V = OUT / 'visual_review'
A = OUT / 'audit'
U = OUT / 'upload'

for path in [OUT, T, D, V, A, U]:
    path.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)

## 1. Resolve and verify Notebook 53 and Notebook 54

In [ ]:
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else latest_complete(
    N53_ROOT,
    [
        'stage_f_verdict.json',
        'experiment_identity.json',
        'tables/page_equivalence.csv',
        'tables/mask_inventory.csv',
        'tables/regional_masks.csv',
        'pages',
    ]
)

N54 = (
    Path(NOTEBOOK54_RUN_OVERRIDE) if NOTEBOOK54_RUN_OVERRIDE else latest_complete(
        N54_ROOT,
        [
            'notebook54_verdict.json',
            'tables/stage_results.csv',
            'tables/exact_equivalence_comparisons.csv',
            'tables/internal_contracts_by_page.csv',
        ],
    )
)

require(N53 is not None, 'Complete Notebook 53R3 run not found.')
require(N54 is not None, 'Complete Notebook 54 run not found.')

v53=read_json(N53/'stage_f_verdict.json')
v54=read_json(N54/'notebook54_verdict.json')

require(
    v53.get('verdict') == 'PASS_STAGE_F_REGRESSION_COMPATIBILITY',
    'Notebook 53R3 did not pass regression compatibility.'
)

require(
    v54.get('verdict') ==
    'PASS_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE_READY_FOR_NOTEBOOK09_SHADOW',
    'Notebook 54 did not pass registered-replay exact equivalence.'
)

require(
    v54.get('first_failed_stage') is None,
    'Notebook 54 records a failed stage.'
)

require(
    v54.get('automatic_output_authority') is False,
    'Notebook 54 has unexpected automatic authority.'
)

id53=read_json(N53/'experiment_identity.json')

require(
    id53.get('version') ==
    '53R3-v8.8b-notebook51R2-regression-compatibility',
    'Unexpected Notebook 53R3 version.'
)
require(
    id53.get('mode') == 'REGISTERED_REPLAY',
    'Notebook 53R3 is not registered replay.'
)

source_version=id53['version']
source_mode=id53['mode']

print('N53R3:',N53)
print('N54:',N54)
print('Source version:',source_version)
print('Mode:',source_mode)

## 2. Resolve the incumbent Notebook 09 approved/final pages

In [ ]:
if INCUMBENT_PAGE_DIR_OVERRIDE is not None:
    incumbent_dir=Path(INCUMBENT_PAGE_DIR_OVERRIDE); N09=incumbent_dir.parent
else:
    N09=Path(NOTEBOOK09_RUN_OVERRIDE) if NOTEBOOK09_RUN_OVERRIDE else latest_complete(N09_ROOT,['pilot_manifest.json','final'])
    require(N09 is not None,'No completed Notebook 09 run found. Set NOTEBOOK09_RUN_OVERRIDE or INCUMBENT_PAGE_DIR_OVERRIDE.')
    incumbent_dir=N09/'final'
require(incumbent_dir.exists(),f'Incumbent page directory missing: {incumbent_dir}')
incumbent=page_files(incumbent_dir)
v88={p.name: p/'corrected.png' for p in (N53/'pages').iterdir() if p.is_dir() and (p/'corrected.png').exists()}
roster53=set(v88); roster09=set(incumbent)
roster_audit={'notebook53_pages':len(roster53),'notebook09_pages':len(roster09),'only_notebook53':sorted(roster53-roster09),'only_notebook09':sorted(roster09-roster53),'exact':roster53==roster09}
atomic_json(A/'roster_audit.json',roster_audit)
require(roster53==roster09,f'Shadow roster mismatch: {roster_audit}')
pages=sorted(roster53); print('Shadow pages:',len(pages),'Notebook09:',N09)


## 3. Pixel-exact shadow comparison and disagreement localisation

In [ ]:
rows = []

for page in pages:
    notebook09 = imread(incumbent[page])
    replay = imread(v88[page])

    observed = imread(
        N53 / 'pages' / page / 'observed.png'
    )

    m3 = (
        imread(N53 / 'pages' / page / 'm3.png')
        > 0
    )

    safe_remove = (
        imread(
            N53 / 'pages' / page / 'safe_remove.png'
        )
        > 0
    )

    safe_protect = (
        imread(
            N53
            / 'pages'
            / page
            / 'safe_protect_decision.png'
        )
        > 0
    )

    m5 = (
        imread(N53 / 'pages' / page / 'm5.png')
        > 0
    )

    shapes = {
        'notebook09': notebook09.shape,
        'registered_replay': replay.shape,
        'observed': observed.shape,
        'm3': m3.shape,
        'safe_remove': safe_remove.shape,
        'safe_protect': safe_protect.shape,
        'm5': m5.shape,
    }

    require(
        len(set(shapes.values())) == 1,
        f'{page}: shape mismatch: {shapes}'
    )

    expected_m5 = (
        m3
        & ~safe_remove
        & ~safe_protect
    )

    require(
        np.array_equal(m5, expected_m5),
        f'{page}: 53R3 registered-replay M5 '
        f'contract failure'
    )

    require(
        not np.any(safe_remove & ~m3),
        f'{page}: SAFE_REMOVE outside M3'
    )

    require(
        not np.any(safe_protect & ~m3),
        f'{page}: SAFE_PROTECT decision outside M3'
    )

    require(
        not np.any(safe_remove & safe_protect),
        f'{page}: decision-class overlap'
    )

    diff = notebook09 != replay

    absdiff = np.abs(
        notebook09.astype(np.int16)
        - replay.astype(np.int16)
    )

    outside_m3 = int(np.sum(diff & ~m3))
    inside_m3 = int(np.sum(diff & m3))
    on_safe_remove = int(np.sum(diff & safe_remove))
    on_safe_protect = int(np.sum(diff & safe_protect))
    on_m5 = int(np.sum(diff & m5))

    classified_union = (
        safe_remove
        | safe_protect
        | m5
    )

    unclassified_inside_m3 = int(np.sum(
        diff
        & m3
        & ~classified_union
    ))

    difference_dir = D / page
    difference_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    locator = np.zeros_like(notebook09)
    locator[diff] = 255

    cv2.imwrite(
        str(
            difference_dir
            / 'difference_locator.png'
        ),
        locator,
    )

    cv2.imwrite(
        str(
            difference_dir
            / 'absolute_difference.png'
        ),
        np.clip(
            absdiff,
            0,
            255,
        ).astype(np.uint8),
    )

    overlay = cv2.cvtColor(
        observed,
        cv2.COLOR_GRAY2BGR,
    )

    overlay[diff] = (0, 0, 255)

    cv2.imwrite(
        str(
            difference_dir
            / 'difference_overlay.png'
        ),
        overlay,
    )

    rows.append(
        {
            'page': page,
            'pixels': int(notebook09.size),
            'different_pixels': int(diff.sum()),
            'different_fraction': float(diff.mean()),
            'max_absolute_difference':
                int(absdiff.max()),
            'mean_absolute_difference_on_diff':
                float(absdiff[diff].mean())
                if diff.any()
                else 0.0,
            'outside_M3_disagreement':
                outside_m3,
            'inside_M3_disagreement':
                inside_m3,
            'SAFE_REMOVE_disagreement':
                on_safe_remove,
            'SAFE_PROTECT_DECISION_disagreement':
                on_safe_protect,
            'M5_REGISTERED_REPLAY_disagreement':
                on_m5,
            'unclassified_inside_M3_disagreement':
                unclassified_inside_m3,
            'exact':
                bool(not diff.any()),
            'notebook09_sha256':
                array_hash(notebook09),
            'registered_replay_sha256':
                array_hash(replay),
            'observed_sha256':
                array_hash(observed),
            'm3_sha256':
                array_hash(m3.astype(np.uint8)),
            'safe_remove_sha256':
                array_hash(
                    safe_remove.astype(np.uint8)
                ),
            'safe_protect_sha256':
                array_hash(
                    safe_protect.astype(np.uint8)
                ),
            'm5_sha256':
                array_hash(m5.astype(np.uint8)),
        }
    )

compare = (
    pd.DataFrame(rows)
    .sort_values(
        ['different_pixels', 'page'],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

compare.to_csv(
    T / 'shadow_page_comparison.csv',
    index=False,
)

require(
    len(compare) == len(pages),
    'Not every page was compared.'
)

outside_m3_total = int(
    compare.outside_M3_disagreement.sum()
)

if outside_m3_total:
    print(
        'WARNING: shadow outputs differ outside M3 at '
        f'{outside_m3_total} pixels. This is diagnostic '
        'and does not establish an outside-authority '
        'violation.'
    )

print(
    compare[
        [
            'page',
            'different_pixels',
            'outside_M3_disagreement',
            'SAFE_REMOVE_disagreement',
            'SAFE_PROTECT_DECISION_disagreement',
            'M5_REGISTERED_REPLAY_disagreement',
            'unclassified_inside_M3_disagreement',
        ]
    ]
    .head(20)
    .to_string(index=False)
)

## 4. Review queues and contact sheets

In [ ]:
compare.assign(review_priority=np.where(compare.different_pixels>0,'REVIEW','NO_DIFFERENCE')).to_csv(T/'shadow_review_queue.csv',index=False)
def sheet(frame,name,n=12):
    q=frame.head(n)
    if q.empty:return
    fig,ax=plt.subplots(len(q),3,figsize=(15,max(5,5*len(q))),squeeze=False)
    for r,(_,row) in enumerate(q.iterrows()):
        page=row.page
        ims=[imread(incumbent[page]),imread(v88[page]),imread(D/page/'difference_overlay.png',cv2.IMREAD_COLOR)]
        titles=[f'{page} Notebook 09',f'{page} V8.8',f'{page} disagreement']
        for j,(im,title) in enumerate(zip(ims,titles)):
            if im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            ax[r,j].imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0,vmax=255); ax[r,j].set_title(title); ax[r,j].axis('off')
    plt.tight_layout(); plt.savefig(V/f'{name}.png',dpi=120,bbox_inches='tight'); plt.close(fig)
sheet(compare[compare.different_pixels>0],'largest_shadow_disagreements')
sheet(compare[compare.different_pixels==0],'exact_shadow_matches')


## 5. Fail-closed shadow verdict

In [ ]:
numeric_columns = [
    'pixels',
    'different_pixels',
    'different_fraction',
    'max_absolute_difference',
    'mean_absolute_difference_on_diff',
    'outside_M3_disagreement',
    'inside_M3_disagreement',
    'SAFE_REMOVE_disagreement',
    'SAFE_PROTECT_DECISION_disagreement',
    'M5_REGISTERED_REPLAY_disagreement',
    'unclassified_inside_M3_disagreement',
]

all_outputs_finite = bool(
    np.isfinite(
        compare[numeric_columns].to_numpy(
            dtype=float
        )
    ).all()
)

gates = {
    'notebook53_pass': True,
    'notebook54_pass': True,
    'roster_exact': roster53 == roster09,
    'all_pages_compared':
        len(compare) == len(pages),
    'comparison_outputs_finite':
        all_outputs_finite,
    'registered_replay_m5_contract_checked':
        True,
}

first_failed=next((k for k,v in gates.items() if v is not True),None)
verdict='PASS_REGISTERED_REPLAY_NOTEBOOK09_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT' if first_failed is None else 'FAIL_V8_8D_REGISTERED_REPLAY_NOTEBOOK09_SHADOW'
record={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'verdict':verdict,
    'first_failed_gate':first_failed,
    'gates':gates,
    'notebook53_version': source_version,
    'notebook53_mode': source_mode,
    'historical_registered_replay': True,
    'historical_code2_exclusion_replayed': True,
    'safe_protect_operationally_authorised': False,
    'automatic_output_authority':False,
    'meaning':'PASS permits Notebook 56 supervised pilot. Notebook 09 remains decision authority.'
}

atomic_json(OUT/'notebook55_verdict.json',record); print(json.dumps(record,indent=2))

if first_failed: raise RuntimeError(f'Notebook 55 failed at {first_failed}')


## 6. Curated upload package

In [ ]:
summary = [
    'NOTEBOOK 55 / V8.8D NOTEBOOK 09 SHADOW COMPARISON',
    '',
    f'Verdict: {verdict}',
    f'Pages: {len(compare)}',
    (
        'Pages with disagreement: '
        f'{int((compare.different_pixels > 0).sum())}'
    ),
    (
        'Total disagreement pixels: '
        f'{int(compare.different_pixels.sum())}'
    ),
    (
        'Outside-M3 diagnostic disagreement pixels: '
        f'{int(compare.outside_M3_disagreement.sum())}'
    ),
    f'Notebook 53 version: {source_version}',
    f'Notebook 53 mode: {source_mode}',
    'Historical code-2 exclusion replayed: True',
    'SAFE_PROTECT operationally authorised: False',
    '',
    (
        'Notebook 09 remained decision authority. '
        'No threshold, model or policy mutation occurred.'
    ),
]

record = {
    'created_utc':
        datetime.now(timezone.utc).isoformat(),

    'verdict':
        verdict,

    'first_failed_gate':
        first_failed,

    'gates':
        gates,

    'notebook53_run':
        str(N53),

    'notebook54_run':
        str(N54),

    'notebook09_run':
        str(N09),

    'incumbent_page_directory':
        str(incumbent_dir),

    'notebook53_version':
        source_version,

    'notebook53_mode':
        source_mode,

    'notebook53_verdict_sha256':
        sha256_file(
            N53 / 'stage_f_verdict.json'
        ),

    'notebook54_verdict_sha256':
        sha256_file(
            N54 / 'notebook54_verdict.json'
        ),

    'historical_registered_replay':
        True,

    'historical_code2_exclusion_replayed':
        True,

    'safe_protect_operationally_authorised':
        False,

    'automatic_output_authority':
        False,

    'outside_m3_is_diagnostic_only':
        True,

    'meaning': (
        'PASS permits Notebook 56 supervised pilot. '
        'Notebook 09 remains decision authority.'
    ),
}
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))
files=[OUT/'executive_summary.txt',OUT/'notebook55_verdict.json',A/'roster_audit.json',T/'shadow_page_comparison.csv',T/'shadow_review_queue.csv']+[p for p in V.glob('*.png')]
rows=[]
for src in files:
    require(src.exists(),f'Missing upload artefact {src}'); dst=U/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(U/'upload_manifest.csv',index=False); (U/'README.txt').write_text('Notebook 55 V8.8D shadow evidence. Notebook 09 remained authoritative.'+chr(10))
zp=U/'notebook55_v8_8d_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in U.iterdir():
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.stat().st_size>0,'Upload ZIP failed'); print('Upload:',zp)
